# FingerBall Stage 2I-2 — MANO 3D hand mesh fitting (single camera, release focus)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atstuyuki/FingerBall-Analysis/blob/main/notebooks/FingerBall_Stage2I2_MANO_3D_Mesh_Fitting.ipynb)

**目的**：Stage 2Hで目視確認済みの手指21点（185–198フレーム）を**一切上書きせず**、左手MANOの形状・姿勢を時系列で推定する。

- 入力：`hand_21_landmarks_2d_corrected.csv`＋必要に応じて元MP4。
- 観測重み：手動修正1.0、目視確認済み引継ぎ候補0.65、AI点0.35、遮蔽・欠損0。
- MANO 16関節＋5つの指先頂点をMediaPipe順21点へ並べ替え。コードで関節対応を検証。
- 目的関数：**2D再投影損失（Huber）＋PCA姿勢事前分布＋形状事前分布＋時間連続性**。単眼で奥行き・スケールは一意に決まらないため、**弱透視（weak-perspective）投影のモデル依存3D**。
- 出力：メッシュHTMLビューア、2D重畳品質管理PNG、再投影誤差CSV、21点3D CSV、関節2D/3D比較CSV、フィットパラメータ（.npz）、設定JSON。**モデル本体やMANOの.pklは出力・再配布しません**。

### 重要：MANOモデルのライセンス
MANOモデル本体は登録と許諾が必要です。[MANO公式サイト](https://mano.is.tue.mpg.de/)でご自身の研究目的に適合するライセンスを確認し、`MANO_LEFT.pkl`を取得してください。公式モデルファイルはGitHubや共有ZIPへアップロードしないでください。公開物にメッシュ等の派生データを含める場合も、許諾条件をご確認ください。

**本Notebookは探索的モデルフィットです。推定された関節角度や奥行きは実測値ではありません。**

## 0. Python環境・Google Drive

In [ ]:
%pip -q install "smplx>=0.1.28" "plotly>=5.18" "pandas>=2.0" "scipy>=1.10" "matplotlib>=3.7" "opencv-python-headless>=4.8"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from IPython.display import display
import os,json,hashlib,math
import numpy as np,pandas as pd
import matplotlib.pyplot as plt
import torch,smplx,cv2
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
print('Torch',torch.__version__,'smplx',getattr(smplx,'__version__','installed'))
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',DEVICE)

## 1. 入力設定と凍結データの読み込み

MANOモデルを `/content/drive/MyDrive/母指回内トラッキング/mano_models/MANO_LEFT.pkl` に配置してください。公式ダウンロードしたモデルを複製・再配布しません。`INPUT_CSV`はStage 2Hの確定CSVです。今回手動確認された`propagated_suggestion`は推定候補であるという由来を残して利用します。

In [ ]:
PROJECT=Path('/content/drive/MyDrive/母指回内トラッキング')
VIDEO_STEM='7A5812501DC5F7120DF842D06EE1B74C695459F1'
INPUT_CSV=PROJECT/'FingerBall_v2_stage2h_results'/VIDEO_STEM/'hand_21_landmarks_2d_corrected.csv'
MANO_MODEL_ROOT=PROJECT/'mano_models'  # MANO_LEFT.pkl in this directory
ORIGINAL_VIDEO=PROJECT/(VIDEO_STEM+'.mp4') # optional QC overlays
OUT_DIR=PROJECT/'FingerBall_v2_stage2i2_results'/VIDEO_STEM
F_START,F_END=185,198
RELEASE_REFERENCE_FRAME=190
HAND='left'
PCA_COMPONENTS=12
STEPS_INITIAL=300
STEPS_REFINE=500
LEARNING_RATE=0.035
RANDOM_SEED=20261031
STATUS_WEIGHTS={'manual_visible':1.0,'propagated_suggestion':0.65,'ai':0.35,
                'interpolated':0.25,'occluded':0.0,'missing':0.0,'outside_edit_window':0.0}
if HAND!='left':raise ValueError('この投球データは左手として確定されています')
if not INPUT_CSV.is_file():raise FileNotFoundError('Stage 2H確定CSVが見つかりません：'+str(INPUT_CSV))
model_file=Path(MANO_MODEL_ROOT)/'MANO_LEFT.pkl'
if not model_file.is_file():
    raise FileNotFoundError(
      'MANO_LEFT.pkl がありません。 https://mano.is.tue.mpg.de/ で登録・許諾確認して取得し、'
      +str(model_file)+' に置いてください。モデル本体はGitHubに置かないでください。')
OUT_DIR.mkdir(parents=True,exist_ok=True)
df=pd.read_csv(INPUT_CSV)
required={'frame','landmark_id','x_px','y_px','annotation_status'}
if missing:=required-set(df.columns):raise ValueError('CSVの不足列: '+str(missing))
if df.duplicated(['frame','landmark_id']).any():raise ValueError('CSVに重複キーがあります')
df=df[df.frame.between(F_START,F_END)].copy()
frames=np.arange(F_START,F_END+1)
if not all(df.groupby('frame').landmark_id.nunique().reindex(frames,fill_value=0).eq(21)):
    raise ValueError('185–198フレームの各21点が必要です')
df['frame']=df.frame.astype(int);df['landmark_id']=df.landmark_id.astype(int)
W,H=1206,648
if 'x_norm' in df and 'y_norm' in df:
    for px,normalized,extent in [('x_px','x_norm',W),('y_px','y_norm',H)]:
        part=df[[px,normalized]].dropna()
        part=part[part[normalized].abs()>.001]
        if len(part)>10 and abs(float(np.median(part[px]/part[normalized]))-extent)>3:
            raise ValueError('入力CSVと1206x648の画像座標が一致しません')
original_hash=hashlib.sha256(INPUT_CSV.read_bytes()).hexdigest()
print('Frozen CSV SHA256:',original_hash)
raw_xy=np.full((len(frames),21,2),np.nan,np.float32)
weights=np.zeros((len(frames),21),np.float32)
status_matrix=np.empty((len(frames),21),dtype=object)
for ti,fr in enumerate(frames):
    block=df[df.frame==fr].set_index('landmark_id').reindex(range(21))
    coords=block[['x_px','y_px']].to_numpy(np.float32)
    stat=block.annotation_status.fillna('missing').to_list()
    if any(x not in STATUS_WEIGHTS for x in stat):
        raise ValueError('Unknown annotation status: '+str(set(stat)-set(STATUS_WEIGHTS)))
    raw_xy[ti]=coords
    weights[ti]=[STATUS_WEIGHTS[x] if np.isfinite(coords[j]).all() else 0.
                 for j,x in enumerate(stat)]
    status_matrix[ti]=stat
obs_normalized=np.nan_to_num((raw_xy-np.array([W/2,H/2],np.float32))/W,nan=0.0)
obs=torch.tensor(obs_normalized,device=DEVICE)
w=torch.tensor(weights,device=DEVICE)
if float(w.sum())<1:raise ValueError('有効な観測ランドマークがありません')
print('Frames:',len(frames),'Weighted points:',round(float(w.sum()),2),
      'Statuses:',pd.Series(status_matrix.ravel()).value_counts().to_dict())

## 2. MANOモデル構築・MediaPipe21点への対応

`smplx`のMANO関節は16点です。指先は公式実装の頂点ID（thumb 744, index 320, middle 443, ring 554, pinky 671）から追加します。**MANO関節配列→MediaPipe配列への変換**を必ず経由します。

In [ ]:
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
T=len(frames)
try:
    mano=smplx.create(str(MANO_MODEL_ROOT),model_type='mano',is_rhand=False,
         use_pca=True,num_pca_comps=PCA_COMPONENTS,flat_hand_mean=True,
         num_betas=10,create_global_orient=False,create_hand_pose=False,
         create_betas=False,create_transl=False,batch_size=T).to(DEVICE)
except Exception as err:
    raise RuntimeError('MANOモデル読み込みに失敗しました。公式MANO_LEFT.pklか、smplx互換性を確認してください') from err
for param in mano.parameters():param.requires_grad_(False)
# smplx MANO 16 joints: wrist, index 3, middle 3, pinky 3, ring 3, thumb 3.
# Append vertices in order thumb/index/middle/ring/pinky, then reorder into MediaPipe 21.
TIP_VERTEX_IDS=[744,320,443,554,671]
MANO_TO_MEDIAPIPE=[0,13,14,15,16,1,2,3,17,4,5,6,18,10,11,12,19,7,8,9,20]
assert len(MANO_TO_MEDIAPIPE)==21 and set(MANO_TO_MEDIAPIPE)==set(range(21))
FACES=np.asarray(mano.faces,dtype=np.int32)
def run_mano(global_orient,hand_pose,betas):
    result=mano(global_orient=global_orient,hand_pose=hand_pose,
                betas=betas.expand(T,-1),return_verts=True)
    vert=result.vertices
    joints=torch.cat([result.joints,vert[:,TIP_VERTEX_IDS]],dim=1)
    if joints.shape[1]!=21:raise RuntimeError(f'MANO joints unexpected: {joints.shape}')
    return vert,joints[:,MANO_TO_MEDIAPIPE,:]
with torch.no_grad():
    v0,j0=run_mano(torch.zeros((T,3),device=DEVICE),
                   torch.zeros((T,PCA_COMPONENTS),device=DEVICE),
                   torch.zeros((1,10),device=DEVICE))
print('MANO mesh:',tuple(v0.shape),'21 joints:',tuple(j0.shape),'faces:',FACES.shape)
print('MANO left model ready')

## 3. 弱透視カメラと正則化付き最適化

- 先に各フレームの手の位置と回転・PCA姿勢を求め、後段で**同一人物の共通形状betas**を推定します。
- 2D誤差にはHuber損失。遮蔽点は重み0、手動点のXYは**入力のまま一切書き換えません**。
- 時間正則化は手指運動を過度に平滑化し得るため、結果は後段QCで比較します。
- カメラの絶対距離・倍率は未校正です。出力されるXYZは**MANOモデル座標系**であり、投球実空間の計測単位を保証しません。

In [ ]:
# Initialization from observed image size and the default MANO pose
obs_np=obs.detach().cpu().numpy()
with torch.no_grad():
    p0=j0.detach().cpu().numpy()[:,:,:2]
    extent_3d=float(max(np.nanmax(np.ptp(p0[:,:,0],axis=1)),np.nanmax(np.ptp(p0[:,:,1],axis=1)),1e-5))
    extents=[]
    for k in range(T):
        use=weights[k]>.0
        pts=obs_np[k,use]
        if len(pts)>1:
            extents.append(max(float(np.ptp(pts[:,0])),float(np.ptp(pts[:,1])),1e-3))
    initial_scale=float(np.median(extents)/extent_3d) if extents else .4
    initial_scale=np.clip(initial_scale,.03,4.)
    initial_centers=[]
    for k in range(T):
        visible=weights[k]>.0
        initial_centers.append(np.median(obs_np[k,visible],axis=0) if visible.any() else [0.,0.])
print('Initial weak-perspective scale:',round(initial_scale,3))
rot=torch.nn.Parameter(torch.zeros(T,3,device=DEVICE))
pca=torch.nn.Parameter(torch.zeros(T,PCA_COMPONENTS,device=DEVICE))
betas=torch.nn.Parameter(torch.zeros(1,10,device=DEVICE))
shift=torch.nn.Parameter(torch.tensor(np.array(initial_centers,np.float32),device=DEVICE))
log_scale=torch.nn.Parameter(torch.full((T,1),math.log(initial_scale),device=DEVICE))
def camera_project(joints):
    # image X to right, image Y downward; MANO local Y upward by convention here.
    u=torch.stack([joints[...,0],-joints[...,1]],dim=-1)
    return torch.exp(log_scale)[:,None,:]*u+shift[:,None,:]
def huber(err,delta=0.008):
    absolute=err.abs()
    return torch.where(absolute<=delta,.5*err.square()/delta,absolute-.5*delta)
def calculate_loss(stage=1):
    verts,joints=run_mano(rot,pca,betas)
    projected=camera_project(joints)
    perpt=huber(projected-obs).sum(dim=-1)
    reproject=(perpt*w).sum()/(w.sum()+1e-8)
    # Per-frame prior: PCA pose encodes joint articulation; rotation is unpenalized except time coherence.
    pose_prior=pca.square().mean()
    shape_prior=betas.square().mean()
    temporal_pose=(pca[1:]-pca[:-1]).square().mean()
    temporal_rot=(rot[1:]-rot[:-1]).square().mean()
    temporal_cam=(shift[1:]-shift[:-1]).square().mean()
    temporal_scale=(log_scale[1:]-log_scale[:-1]).square().mean()
    # Observations dominate the regularizers. Check reprojection residuals after fit.
    total=(reproject + .0002*pose_prior + .00001*shape_prior
         + .0005*temporal_pose + .00005*temporal_rot
         + .005*temporal_cam + .0001*temporal_scale)
    return total,dict(reproj=reproject,pose=pose_prior,temporal=temporal_pose),verts,joints,projected
history=[]
def train(steps,stage=1):
    optim_params=[rot,pca,shift,log_scale]+([betas] if stage==2 else [])
    optimizer=torch.optim.Adam(optim_params,lr=LEARNING_RATE if stage==1 else LEARNING_RATE*.5)
    for it in range(steps):
        optimizer.zero_grad()
        loss,parts,_,_,_=calculate_loss(stage)
        if not torch.isfinite(loss):raise FloatingPointError('MANO loss NaN/Inf; adjust LR or input')
        loss.backward()
        torch.nn.utils.clip_grad_norm_(optim_params,max_norm=1.5)
        optimizer.step()
        with torch.no_grad():
            rot.clamp_(-3.2,3.2)
            pca.clamp_(-5.,5.)
            betas.clamp_(-2.,2.)
            log_scale.clamp_(math.log(.01),math.log(15.))
        if it%25==0 or it==steps-1:
            entry={'stage':stage,'step':it,'loss':float(loss.detach()),
                   'reproj_loss':float(parts['reproj'].detach())}
            history.append(entry)
            if it%100==0 or it==steps-1:print(entry)
train(STEPS_INITIAL,stage=1)
train(STEPS_REFINE,stage=2)
with torch.no_grad():
    loss,parts,final_vertices,final_joints,projected=calculate_loss(2)
    meshes=final_vertices.cpu().numpy()
    joints3d=final_joints.cpu().numpy()
    fit_xy=projected.cpu().numpy()*W+np.array([W/2,H/2],np.float32)
print('Final weighted reprojection loss:',float(parts['reproj']))
history_df=pd.DataFrame(history)
history_df.to_csv(OUT_DIR/'fit_optimization_history.csv',index=False)

## 4. 再投影誤差と関節座標の保存

手動座標を確定値として保持し、**再投影座標を別列として記録**します。メッシュの再投影位置が一致しないときは、MANO 3Dの定量結果を採用しないでください。

In [ ]:
joint_names=['wrist','thumb_cmc','thumb_mcp','thumb_ip','thumb_tip',
             'index_mcp','index_pip','index_dip','index_tip',
             'middle_mcp','middle_pip','middle_dip','middle_tip',
             'ring_mcp','ring_pip','ring_dip','ring_tip',
             'pinky_mcp','pinky_pip','pinky_dip','pinky_tip']
joint_rows=[]
for k,f in enumerate(frames):
    for i in range(21):
        target=raw_xy[k,i]
        pred=fit_xy[k,i]
        visible=np.isfinite(target).all()
        err=float(np.linalg.norm(pred-target)) if visible else np.nan
        joint_rows.append({'frame':int(f),'landmark_id':i,'joint_name':joint_names[i],
          'annotation_status':status_matrix[k,i],'fit_weight':float(weights[k,i]),
          'target_x_px':float(target[0]) if visible else np.nan,
          'target_y_px':float(target[1]) if visible else np.nan,
          'reprojected_x_px':float(pred[0]),'reprojected_y_px':float(pred[1]),
          'reprojection_error_px':err,
          'mano_x':float(joints3d[k,i,0]),'mano_y':float(joints3d[k,i,1]),
          'mano_z':float(joints3d[k,i,2])})
fit_table=pd.DataFrame(joint_rows)
fit_table.to_csv(OUT_DIR/'mano_fit_joint21_with_provenance.csv',index=False)
frame_qc=[]
for f,group in fit_table.groupby('frame'):
    usable=group[group.fit_weight>0]
    manual=group[group.annotation_status=='manual_visible']
    frame_qc.append({'frame':int(f),'n_weighted':len(usable),'n_manual':len(manual),
       'weighted_RMSE_px':float(np.sqrt(np.average(
         usable.reprojection_error_px.to_numpy()**2,
         weights=usable.fit_weight.to_numpy()))) if len(usable) else np.nan,
       'manual_RMSE_px':float(np.sqrt(np.mean(manual.reprojection_error_px.to_numpy()**2))) if len(manual) else np.nan,
       'max_observed_error_px':float(usable.reprojection_error_px.max()) if len(usable) else np.nan})
qc=pd.DataFrame(frame_qc)
qc.to_csv(OUT_DIR/'mano_fit_frame_reprojection_qc.csv',index=False)
print(qc.round(2).to_string(index=False))
print('Weighted overall RMSE (px):',round(float(np.sqrt(np.average(
 fit_table.loc[fit_table.fit_weight>0,'reprojection_error_px']**2,
 weights=fit_table.loc[fit_table.fit_weight>0,'fit_weight']))),2))

## 5. モデル依存の3D関節内角と、2D投影角との比較

**重要：ここでの3D角度は3点のMANOモデル関節内角**であり、解剖学的関節軸で定義した屈伸角とは異なります。絶対角度を臨床測定値として使用しないでください。

In [ ]:
ANGLE_TRIPLES={'index_PIP':(5,6,7),'index_DIP':(6,7,8),
               'middle_PIP':(9,10,11),'middle_DIP':(10,11,12),
               'thumb_IP':(2,3,4)}
def included_angle(coords,a,b,c):
    u=coords[a]-coords[b]
    v=coords[c]-coords[b]
    if not (np.isfinite(u).all() and np.isfinite(v).all()):return np.nan
    denom=float(np.linalg.norm(u)*np.linalg.norm(v))
    if denom<1e-10:return np.nan
    return float(np.degrees(np.arccos(np.clip(float(np.dot(u,v))/denom,-1.,1.))))
kin_rows=[]
for k,f in enumerate(frames):
    record={'frame':int(f)}
    for name,(a,b,c) in ANGLE_TRIPLES.items():
        record[name+'_original_2D_deg']=included_angle(raw_xy[k],a,b,c)
        record[name+'_reprojected_2D_deg']=included_angle(fit_xy[k],a,b,c)
        record[name+'_MANO_3D_included_deg']=included_angle(joints3d[k],a,b,c)
        record[name+'_all_manual']=bool(all(status_matrix[k,i]=='manual_visible' for i in (a,b,c)))
    kin_rows.append(record)
kin=pd.DataFrame(kin_rows)
kin.to_csv(OUT_DIR/'mano_2D_vs_3D_included_angles.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(14,4.5))
for finger,ax in [('index',axes[0]),('middle',axes[1])]:
    for joint in ('PIP','DIP'):
        key=f'{finger}_{joint}'
        ax.plot(kin.frame,kin[key+'_original_2D_deg'],marker='o',label=key+' observed 2D')
        ax.plot(kin.frame,kin[key+'_MANO_3D_included_deg'],linestyle='--',alpha=.7,label=key+' MANO 3D')
    ax.axvline(RELEASE_REFERENCE_FRAME,color='black',linestyle=':',label='visual release reference')
    ax.set(title=finger.title(),xlabel='Frame',ylabel='3-point included angle (deg)')
    ax.grid(alpha=.25);ax.legend(fontsize=8)
fig.suptitle('FingerBall 2D observations versus model-dependent 3D angles')
fig.tight_layout()
fig.savefig(OUT_DIR/'mano_2D_vs_3D_angles.png',dpi=160)
plt.show()

## 6. 画像上での2D再投影検証とMANO 3Dメッシュビューア

188〜193フレームを中心に、手の位置・形状が**元動画および目視確認済み21点**に沿っているか確かめます。元動画が見つからない場合、2D図は座標だけ表示。HTMLは**自身の研究用Drive**に保存します。

In [ ]:
# 2D reference: observed vs MANO reprojection. Original video is optional.
view_frames=[int(f) for f in frames if 188<=f<=193]
cap=cv2.VideoCapture(str(ORIGINAL_VIDEO)) if ORIGINAL_VIDEO.is_file() else None
if cap is not None and not cap.isOpened():
    cap.release();cap=None
def render_2d_frame(f):
    k=int(np.where(frames==f)[0][0])
    image=None
    if cap is not None:
        cap.set(cv2.CAP_PROP_POS_FRAMES,f)
        ok,bgr=cap.read()
        if ok:image=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)
    fig,ax=plt.subplots(figsize=(10,6))
    if image is not None:ax.imshow(image)
    else:ax.set(xlim=(0,W),ylim=(H,0))
    use=np.isfinite(raw_xy[k]).all(axis=1)&(weights[k]>0)
    ax.scatter(raw_xy[k,use,0],raw_xy[k,use,1],c='#f59e0b',s=22,
               label='Stage 2H observed 2D',zorder=3)
    ax.scatter(fit_xy[k,:,0],fit_xy[k,:,1],c='#06b6d4',s=16,
               marker='x',label='MANO reprojection',zorder=4)
    for i in np.where(use)[0]:
        ax.plot([raw_xy[k,i,0],fit_xy[k,i,0]],
                [raw_xy[k,i,1],fit_xy[k,i,1]],color='#e879f9',alpha=.45,lw=.8)
    coords=raw_xy[k,use]
    if len(coords):
        c=coords.mean(axis=0)
        ax.set_xlim(max(0,c[0]-110),min(W,c[0]+110))
        ax.set_ylim(min(H,c[1]+110),max(0,c[1]-110))
    ax.set(title=f'Frame {f}: frozen 2D (orange) vs fitted MANO (cyan)',
           xlabel='Original X pixels',ylabel='Original Y pixels')
    ax.legend(loc='upper right',fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT_DIR/f'mano_reprojection_frame_{f}.png',dpi=160)
    plt.show()
for f in view_frames:render_2d_frame(f)
if cap is not None:cap.release()

In [ ]:
# Standalone interactive 3D mesh, 188–193. Contains MANO-derived geometry:
# do not redistribute without confirming the MANO license conditions.
if not view_frames:raise RuntimeError('No release frames for viewer')
def mesh_trace(k):
    p=meshes[k]
    return go.Mesh3d(x=p[:,0],y=p[:,1],z=p[:,2],
         i=FACES[:,0],j=FACES[:,1],k=FACES[:,2],
         color='#68c4db',opacity=.86,flatshading=False,
         lighting=dict(ambient=.5,diffuse=.8,specular=.1),name='MANO mesh')
def joint_trace(k):
    q=joints3d[k]
    return go.Scatter3d(x=q[:,0],y=q[:,1],z=q[:,2],mode='markers',
        marker=dict(size=3,color='#ffb454'),name='MANO 21 joints')
selected=[int(np.where(frames==f)[0][0]) for f in view_frames]
viewer=go.Figure(data=[mesh_trace(selected[0]),joint_trace(selected[0])],
    frames=[go.Frame(name=str(frames[k]),data=[mesh_trace(k),joint_trace(k)])
            for k in selected])
joint_min=joints3d[selected].reshape(-1,3).min(axis=0)
joint_max=joints3d[selected].reshape(-1,3).max(axis=0)
center=(joint_min+joint_max)/2
span=float(np.max(joint_max-joint_min)*.65)
span=max(span,.025)
viewer.update_layout(
    title='FingerBall: fitted LEFT MANO hand (model-dependent 3D)',
    template='plotly_dark',height=670,
    scene=dict(xaxis=dict(range=[center[0]-span,center[0]+span]),
       yaxis=dict(range=[center[1]-span,center[1]+span]),
       zaxis=dict(range=[center[2]-span,center[2]+span]),
       aspectmode='cube',camera=dict(eye=dict(x=1.6,y=1.6,z=.8))),
    sliders=[dict(currentvalue=dict(prefix='Frame '),steps=[
       dict(label=str(frames[k]),method='animate',args=[[str(frames[k])],
             dict(mode='immediate',frame=dict(duration=0,redraw=True),
                  transition=dict(duration=0))]) for k in selected])],
    updatemenus=[dict(type='buttons',buttons=[
       dict(label='Play',method='animate',args=[None,
         dict(frame=dict(duration=500,redraw=True),fromcurrent=True)]),
       dict(label='Pause',method='animate',args=[[None],
         dict(mode='immediate',frame=dict(duration=0,redraw=True))])])])
viewer.show(config=dict(displaylogo=False))
viewer_path=OUT_DIR/'FingerBall_Stage2I2_MANO_3D_mesh_viewer.html'
pio.write_html(viewer,str(viewer_path),include_plotlyjs=True,full_html=True)
print('Interactive HTML:',viewer_path)

## 7. パラメータ・品質管理レポートの保存と原本確認

**fitパラメータは自分のDriveへ保存し、元のStage2H CSVはSHA256で不変を確認**します。最終的な解析採否は、フレームごとの再投影RMSEと重畳画像を目視確認して決めてください。

In [ ]:
params_path=OUT_DIR/'mano_fit_params.npz'
np.savez_compressed(params_path,
    frame=frames,global_orient=rot.detach().cpu().numpy(),
    hand_pose_pca=pca.detach().cpu().numpy(),betas=betas.detach().cpu().numpy(),
    camera_shift_norm=shift.detach().cpu().numpy(),
    camera_log_scale=log_scale.detach().cpu().numpy(),
    model_joints3d=joints3d,model_vertices=meshes)
source_same=hashlib.sha256(INPUT_CSV.read_bytes()).hexdigest()==original_hash
if not source_same:raise RuntimeError('入力確定CSVが途中で変更されています！')
settings={
    'stage':'2I-2','method':'MANO fit with weak perspective camera',
    'original_csv':str(INPUT_CSV),'input_sha256':original_hash,
    'input_unchanged':True,'model_file_name':model_file.name,
    'hand':'left','frames':[int(frames[0]),int(frames[-1])],
    'pca_components':PCA_COMPONENTS,'initial_steps':STEPS_INITIAL,
    'refine_steps':STEPS_REFINE,'lr':LEARNING_RATE,
    'status_weights':STATUS_WEIGHTS,
    'limitations':[
       'Monocular weak-perspective MANO inference is not ground-truth 3D',
       '3D coordinate axes and scale are MANO model-specific and not calibrated to physical space',
       'MANO 3D included joint angles are not necessarily anatomical flexion angles',
       'Manual 2D targets were not modified, only optimized projected predictions',
       'No independent depth, multiview or external ground truth validation',
       'MANO-derived meshes may be subject to licence restrictions']}
(OUT_DIR/'mano_fit_report.json').write_text(
    json.dumps(settings,ensure_ascii=False,indent=2),encoding='utf-8')
print('Complete output folder:',OUT_DIR)
print('Source CSV unchanged:',source_same)
print('Expected outputs:',
  [p.name for p in sorted(OUT_DIR.iterdir()) if p.is_file()])
print('MANO-derived data and model files must NOT be redistributed without license review.')

## 出力の確認・研究上の注意

**一次評価**：`mano_fit_frame_reprojection_qc.csv`の手動点RMSEと画像重畳を確認すること。悪いフレームでは、3Dモデルの指角度を解釈しないこと。

**次の段階**：フレーム内のモデル姿勢が適切なら、拘束重みの感度分析（手動のみ vs 手動＋引継ぎ候補）と、別の投球動画を用いた外部検証へ進む。今回の単一動画だけで3D精度を論証しないこと。

**引用文献**：Romero et al. *Embodied Hands: Modeling and Capturing Hands and Bodies Together.* ACM TOG (SIGGRAPH Asia), 2017. MANO license: https://mano.is.tue.mpg.de/